# Do motifs add anything over simpler readouts of the LFP and EEG?

Same data as `awake_grounding.ipynb`: 65 region signals, 8 recordings, 7 mice, awake stimulation-free seconds only.
Everything is scored with Mishler's **C1 test**: a per-second signal splits the region's neurons into a + and a −
ensemble (on 4/5 of the data); the score is the held-out correlation between the signal and (+ ens − − ens);
p from 200 circular shifts of the LFP/EEG against the spikes (≥ 5 min). Code: `src/motif_value.py`.

1. **Same size, no spikes used.** Is the motif direction better than other directions found without spikes:
   the band principal components, broadband power, random directions?
2. **Transfer across mice.** Templates learned in the other mice, applied unchanged, vs a band → population-rate
   regression learned in the other mice and applied unchanged.
3. **EEG.** Motifs on the skull EEG electrode nearest the region (and, as a locality control, the farthest one),
   scored against the region's neurons.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src.state_motifs import BANDS
P = REPO / "data" / "processed" / "motif_value"
OUT = REPO / "data" / "out" / "2026-11-15_motif_value"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "0.4", "axes.labelcolor": "0.15", "xtick.color": "0.3", "ytick.color": "0.3"})
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
BAND_LAB = [f"{lo:g}–{hi:g}" for lo, hi in BANDS]
MOTIF, PC, BB, RAND, REG = "#2a78d6", "#eb6834", "#7a7a75", "#c9c9c4", "#6250d6"
rng = np.random.default_rng(0)
def wtest(a, b):
    d = pd.concat([a, b], axis=1).dropna()
    return d.iloc[:, 0].median(), d.iloc[:, 1].median(), f"{(d.iloc[:, 0] > d.iloc[:, 1]).sum()}/{len(d)}", wilcoxon(d.iloc[:, 0], d.iloc[:, 1]).pvalue

D = pd.read_pickle(P / "directions.pkl"); D["sig"] = D.session + " " + D.region
T = pd.read_pickle(P / "transfer.pkl"); T["sig"] = T.session + " " + T.region
CONS = pd.read_pickle(P / "consensus.pkl")
E = pd.read_pickle(P / "eeg.pkl"); E["sig"] = E.session + " " + E.region
EP = pd.read_pickle(P / "eeg_population_removed.pkl"); EP["sig"] = EP.session + " " + EP.region

## 1. Same size, no spikes: motif vs band PCs vs broadband vs random directions
Per signal: the best motif against the best of the same number of top band PCs, against broadband, and its
percentile among 200 random directions. Three frames, each scored on its own.

In [2]:
rows = []
for (fr, sig), d in D.groupby(["frame", "sig"]):
    for ms in [k for k in d.kind.unique() if "motif" in k]:
        mot = d[d.kind == ms]; k = len(mot)
        pcs = d[d.kind == "band PC"].sort_values("direction").head(k)
        best = mot.loc[mot.r_diff.idxmax()] if mot.r_diff.notna().any() else None
        if best is None: continue
        rows.append({"frame": fr, "motif set": ms, "sig": sig, "best motif": best.r_diff, "best PCs (same number)": pcs.r_diff.max(),
                     "PC1": d[d.direction == "PC1"].r_diff.iloc[0], "broadband": d[d.kind == "broadband"].r_diff.iloc[0],
                     "random median": best.random_median, "motif percentile among random": best.random_percentile,
                     "|cos| motif–PC1": best.abs_cos_pc1, "|cos| motif–broadband": best.abs_cos_broadband})
PAIR = pd.DataFrame(rows)
S1 = []
for (fr, ms), d in PAIR.groupby(["frame", "motif set"], sort=False):
    a, b, w, p = wtest(d["best motif"], d["best PCs (same number)"])
    _, c, w2, p2 = wtest(d["best motif"], d["broadband"])
    S1.append({"frame": fr, "motifs": ms, "signals": len(d), "motif": a, "PCs": b, "motif > PCs": w, "p vs PCs": p,
               "broadband": c, "motif > broadband": w2, "p vs broadband": p2, "random (median)": d["random median"].median(),
               "motif percentile among random": d["motif percentile among random"].median(),
               "|cos| motif–PC1": d["|cos| motif–PC1"].median(), "|cos| motif–broadband": d["|cos| motif–broadband"].median()})
S1 = pd.DataFrame(S1); S1.to_csv(OUT / "1_directions_summary.csv", index=False)

panels = [("binarized", "original motif"), ("continuous", "original motif"), ("tracker", "tracker motif")]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (fr, ms) in zip(axes, panels):
    d = PAIR[(PAIR.frame == fr) & (PAIR["motif set"] == ms)]
    cols = [("random median", RAND, "random\n(median of 200)"), ("broadband", BB, "broadband"),
            ("best PCs (same number)", PC, "band PCs\n(same number)"), ("best motif", MOTIF, "motif")]
    M = d[[c for c, _, _ in cols]].to_numpy()
    for row in M: ax.plot(range(4), row, color="0.88", lw=0.7, zorder=1)
    for k, (c, col, lab) in enumerate(cols):
        ax.scatter(np.full(len(d), k), d[c], s=14, color=col, zorder=2)
        ax.plot([k - 0.3, k + 0.3], [d[c].median()] * 2, color="0.1", lw=2, zorder=3)
    ax.set_xticks(range(4), [lab for _, _, lab in cols], fontsize=8)
    s = S1[(S1.frame == fr) & (S1.motifs == ms)].iloc[0]
    ax.set_title(f"{fr} frame, {ms}\nmotif vs PCs: wins {s['motif > PCs']}, p = {s['p vs PCs']:.3f}; "
                 f"|cos| with PC1 = {s['|cos| motif–PC1']:.2f}", fontsize=8.5, loc="left")
axes[0].set_ylabel("held-out C1 score r(signal, + ens − − ens)")
fig.suptitle("1. Directions found without spikes (one line = one signal; bar = median)", fontsize=10, x=0.01, ha="left")
fig.tight_layout(); save(fig, "1_directions"); plt.show()
S1.round(3)

,frame,motifs,signals,motif,PCs,motif > PCs,p vs PCs,broadband,motif > broadband,p vs broadband,random (median),motif percentile among random,|cos| motif–PC1,|cos| motif–broadband
0,binarized,original motif,61,0.625,0.596,32/61,0.387,0.437,49/61,0.0,0.319,0.985,0.993,0.592
1,binarized,relaxed motif,57,0.615,0.606,29/57,0.772,0.446,46/57,0.0,0.324,0.980,0.985,0.556
2,continuous,original motif,61,0.662,0.653,41/61,0.004,0.450,51/61,0.0,0.400,0.970,0.980,0.578
3,continuous,relaxed motif,57,0.663,0.654,35/57,0.326,0.449,45/57,0.0,0.407,0.955,0.969,0.560
4,tracker,tracker motif,42,0.666,0.650,28/42,0.035,0.471,36/42,0.0,0.420,0.970,0.941,0.511


## 2. Transfer across mice
Left: the consensus motif learned with each mouse left out (all seven are near-identical). Right: C1 score and
|r| with the population rate in the held-out mouse, for the mouse's own motif, the transferred consensus motif,
the transferred regression, broadband, and (population rate only) a regression fitted inside the mouse.

In [3]:
def grp(n):
    if n.startswith("own motif"): return "own motif"
    if n.startswith("consensus") and "binarized" in n: return "consensus motif (transferred)"
    if n.startswith("consensus"): return "consensus motif, continuous"
    return n
T["g"] = T.readout.map(grp); T["abs_r_pop"] = T.r_pop.abs()
TB = T.groupby(["sig", "g"]).agg(c1=("c1_r_diff", "max"), pop=("abs_r_pop", "max")).reset_index()
c1 = TB.pivot(index="sig", columns="g", values="c1"); pop = TB.pivot(index="sig", columns="g", values="pop")
ORDER = ["own motif", "consensus motif (transferred)", "transferred regression", "broadband", "in-mouse regression (ceiling)"]
COLS = {"own motif": MOTIF, "consensus motif (transferred)": "#1baf7a", "transferred regression": REG, "broadband": BB,
        "in-mouse regression (ceiling)": "#3a3a37"}
LAB = {"own motif": "own motif", "consensus motif (transferred)": "consensus motif\n(other mice)",
       "transferred regression": "regression\n(other mice)", "broadband": "broadband", "in-mouse regression (ceiling)": "regression\n(same mouse)"}
fig, axes = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [1, 1.2, 1.2]})
ax = axes[0]
for _, rw in CONS.iterrows():
    ax.plot(range(10), rw.template, color="#1baf7a", lw=1, alpha=0.8)
ax.axhline(0, color="0.75", lw=0.6); ax.set_xticks(range(10), BAND_LAB, rotation=45, fontsize=8)
ax.set_ylabel("template weight"); ax.set_title(f"consensus motif, 7 leave-one-mouse-out fits\n"
                                              f"(~{CONS.n_templates.median():.0f} of ~{CONS['of'].median():.0f} motifs agree)", fontsize=8.5, loc="left")
for ax, M, lab in [(axes[1], c1, "held-out C1 score"), (axes[2], pop, "|r| with population rate")]:
    order = [o for o in ORDER if o in M.columns and M[o].notna().any()]
    for row in M[order].to_numpy(): ax.plot(range(len(order)), row, color="0.88", lw=0.7, zorder=1)
    for k, o in enumerate(order):
        ax.scatter(np.full(len(M), k), M[o], s=14, color=COLS[o], zorder=2)
        ax.plot([k - 0.3, k + 0.3], [M[o].median()] * 2, color="0.1", lw=2, zorder=3)
    ax.set_xticks(range(len(order)), [LAB[o] for o in order], fontsize=8); ax.set_title(lab, fontsize=9, loc="left")
fig.suptitle("2. Transfer across mice (one line = one signal; bar = median)", fontsize=10, x=0.01, ha="left")
fig.tight_layout(); save(fig, "2_transfer"); plt.show()
S2 = []
for lab, M in (("C1", c1), ("|r| population rate", pop)):
    for a, b in [("consensus motif (transferred)", "own motif"), ("consensus motif (transferred)", "transferred regression"),
                 ("consensus motif (transferred)", "broadband"), ("transferred regression", "broadband"),
                 ("transferred regression", "in-mouse regression (ceiling)")]:
        if M[a].notna().any() and M[b].notna().any():
            x, y, w, p = wtest(M[a], M[b]); S2.append({"score": lab, "a": a, "b": b, "median a": x, "median b": y, "a wins": w, "p": p})
S2 = pd.DataFrame(S2); S2.to_csv(OUT / "2_transfer_summary.csv", index=False)
CONS.to_csv(OUT / "2_consensus_templates.csv", index=False)
S2.round(4)

,score,a,b,median a,median b,a wins,p
0,C1,consensus motif (transferred),own motif,0.5405,0.6251,13/61,0.0000
1,C1,consensus motif (transferred),transferred regression,0.5361,0.4344,31/65,0.4121
2,C1,consensus motif (transferred),broadband,0.5361,0.4421,37/65,0.1710
3,C1,transferred regression,broadband,0.4344,0.4421,33/65,0.8986
4,|r| population rate,consensus motif (transferred),own motif,0.1607,0.3214,15/61,0.0000
5,|r| population rate,consensus motif (transferred),transferred regression,0.1607,0.3936,18/65,0.0000
6,|r| population rate,consensus motif (transferred),broadband,0.1607,0.2458,28/65,0.4504
7,|r| population rate,transferred regression,broadband,0.3936,0.2458,50/65,0.0000
8,|r| population rate,transferred regression,in-mouse regression (ceiling),0.3936,0.4522,19/65,0.0002


## 3. EEG: do motifs on the skull track the neurons underneath?
Readouts on the nearest valid EEG electrode (median 0.6 mm horizontally from the region's units) and on the
farthest (locality control). EEG own motif = original rules run on that EEG channel. LFP motif = the region's own
LFP motif applied to the EEG bands. Consensus motif = learned from the other mice's LFP. EEG regression = EEG bands
→ population rate, trained on the other mice. Top: raw rates; bottom: population mean removed from every neuron.

In [4]:
def grp3(n):
    return next((k for k in ["EEG own motif", "LFP motif", "consensus motif"] if n.startswith(k)), n)
ORD3 = ["EEG own motif", "LFP motif", "consensus motif", "EEG PC1", "EEG broadband", "EEG regression (other mice)"]
LAB3 = {"EEG own motif": "EEG own\nmotif", "LFP motif": "LFP motif\non EEG", "consensus motif": "consensus\nmotif on EEG",
        "EEG PC1": "EEG PC1", "EEG broadband": "EEG\nbroadband", "EEG regression (other mice)": "EEG regression\n(other mice)"}
S3 = []
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, (lab, X) in zip(axes, [("raw rates", E), ("population mean removed", EP)]):
    X = X.assign(g=X.readout.map(grp3))
    B = X.groupby(["sig", "electrode", "g"]).agg(c1=("c1_r_diff", "max"), p=("c1_p", "min")).reset_index()
    for k, g in enumerate(ORD3):
        for dx, wh, col in ((-0.17, "near", MOTIF if "motif" in g else BB), (0.17, "far", RAND)):
            b = B[(B.g == g) & (B.electrode == wh)]
            ax.scatter(k + dx + rng.uniform(-0.07, 0.07, len(b)), b.c1, s=10, color=col, alpha=0.85)
            ax.plot([k + dx - 0.12, k + dx + 0.12], [b.c1.median()] * 2, color="0.1", lw=2)
        n_ = B[(B.g == g) & (B.electrode == "near")].set_index("sig").c1; f_ = B[(B.g == g) & (B.electrode == "far")].set_index("sig").c1
        a, b2, w, p = wtest(n_, f_)
        ax.text(k, 1.0, f"near > far {w}\np = {p:.3f}", ha="center", fontsize=7.5)
        S3.append({"rates": lab, "readout": g, "near (median)": a, "far (median)": b2, "near wins": w, "p near vs far": p,
                   "near passing": f"{(B[(B.g == g) & (B.electrode == 'near')].p < .05).sum()}/{len(n_)}"})
    ax.set_ylim(-0.3, 1.1); ax.axhline(0, color="0.75", lw=0.6)
    ax.set_ylabel("held-out C1 score"); ax.set_title(f"{lab}: left of each pair = nearest electrode, right (light) = farthest", fontsize=9, loc="left")
axes[1].set_xticks(range(len(ORD3)), [LAB3[g] for g in ORD3], fontsize=8.5)
fig.suptitle("3. EEG readouts scored against the neurons of the region below", fontsize=10, x=0.01, ha="left")
fig.tight_layout(); save(fig, "3_eeg"); plt.show()
S3 = pd.DataFrame(S3); S3.to_csv(OUT / "3_eeg_near_far.csv", index=False)
cmp = []
for lab, X in (("raw rates", E), ("population mean removed", EP)):
    X = X.assign(g=X.readout.map(grp3))
    near = X[X.electrode == "near"].groupby(["sig", "g"]).c1_r_diff.max().unstack()
    for a, b in [("EEG own motif", "EEG broadband"), ("EEG own motif", "EEG PC1"), ("EEG own motif", "EEG regression (other mice)"),
                 ("consensus motif", "EEG regression (other mice)"), ("consensus motif", "EEG broadband")]:
        x, y, w, p = wtest(near[a], near[b]); cmp.append({"rates": lab, "a": a, "b": b, "median a": x, "median b": y, "a wins": w, "p": p})
lfp = D[(D.frame == "binarized") & (D.kind == "original motif")].groupby("sig").r_diff.max()
eegm = E[(E.electrode == "near") & E.readout.str.startswith("EEG own motif")].groupby("sig").c1_r_diff.max()
x, y, w, p = wtest(eegm, lfp); cmp.append({"rates": "raw rates", "a": "EEG own motif (nearest)", "b": "LFP own motif on LFP", "median a": x, "median b": y, "a wins": w, "p": p})
CMP = pd.DataFrame(cmp); CMP.to_csv(OUT / "3_eeg_comparisons.csv", index=False)
display(S3.round(3)); CMP.round(4)

,rates,readout,near (median),far (median),near wins,p near vs far,near passing
0,raw rates,EEG own motif,0.611,0.520,33/47,0.002,49/55
1,raw rates,LFP motif,0.483,0.382,37/61,0.008,55/61
2,raw rates,consensus motif,0.470,0.311,42/65,0.001,57/65
3,raw rates,EEG PC1,0.496,0.441,40/65,0.034,54/65
4,raw rates,EEG broadband,0.400,0.312,43/65,0.130,48/65
5,raw rates,EEG regression (other mice),0.391,0.477,29/65,0.011,49/65
6,population mean removed,EEG own motif,0.526,0.442,30/47,0.003,46/55
7,population mean removed,LFP motif,0.411,0.327,34/61,0.066,51/61
8,population mean removed,consensus motif,0.437,0.315,41/65,0.019,50/65
9,population mean removed,EEG PC1,0.396,0.286,46/65,0.000,47/65


,rates,a,b,median a,median b,a wins,p
0,raw rates,EEG own motif,EEG broadband,0.5470,0.3870,38/55,0.0000
1,raw rates,EEG own motif,EEG PC1,0.5470,0.4829,34/55,0.1059
2,raw rates,EEG own motif,EEG regression (other mice),0.5470,0.3595,45/55,0.0000
3,raw rates,consensus motif,EEG regression (other mice),0.4697,0.3915,42/65,0.0077
4,raw rates,consensus motif,EEG broadband,0.4697,0.4001,36/65,0.0094
5,population mean removed,EEG own motif,EEG broadband,0.4847,0.3418,36/55,0.0000
6,population mean removed,EEG own motif,EEG PC1,0.4847,0.4232,32/55,0.0703
7,population mean removed,EEG own motif,EEG regression (other mice),0.4847,0.2451,41/55,0.0000
8,population mean removed,consensus motif,EEG regression (other mice),0.4368,0.2645,41/65,0.0062
9,population mean removed,consensus motif,EEG broadband,0.4368,0.3432,38/65,0.0098


## Conclusions
1. **Motif ≈ first principal component of the bands.** The leading motif points the same way as band PC1
   (|cos| 0.94–0.99) and scores the same on C1 (binarized: wins 32/61, p = 0.39; continuous and tracker frames:
   a small edge, +0.01–0.02). Both beat broadband (49/61, p < 1e-6) and random directions (motif at the ~97th
   percentile). So the multi-band shape matters (it is not just total power), but the motif algorithm finds the
   same shape that ordinary PCA finds.
2. **One motif is shared across mice:** 1–30 Hz up together with 35–80 Hz down (about a third of all motifs; the
   same in every leave-one-mouse-out fit). Applied unchanged to a new mouse it keeps most of its C1 score (0.54 vs
   0.63 for the mouse's own motif). A regression trained on other mice transfers well for what it was trained on
   (population rate: |r| 0.39 vs 0.45 in-mouse) but no better than broadband at defining ensembles (C1 0.43).
   Regression weights did transfer, so the expected advantage of motifs here is not found.
3. **EEG is where motifs win.** On the nearest skull electrode, the EEG's own motif tracks the neurons below about
   as well as the LFP's motif tracks them from inside the region (C1 0.55 vs 0.63, n.s.); it beats EEG broadband
   (38/55, p < 1e-5) and a regression transferred from other mice (45/55, p < 1e-7), and is on par with EEG PC1
   (34/55, p = 0.11). Motif readouts are stronger on the nearest electrode than on the farthest (p 0.001–0.008),
   also with the population mean removed: part of the signal is local. The transferred EEG regression shows no such
   locality. Much of the score is still shared across the brain: the farthest electrode keeps ~80 % of it.

Overall: motifs do not carry information that PCA or regression lack. Their value is practical: a readout found
without spikes, the same across mice, that works on EEG and is partly local. PCA of the bands shares the first
two properties; whether it also generalises across mice and states as well as motifs is the next comparison.